# Model Training Notebook

This is the Jupyter notebook where both the cross-validation and final training + evaluation were run. The snippets in this notebook call other modules in this directory that did the heavy lifting/automation for these steps (e.g. [keras_utility_classes.py](keras_utility_classes.py)). There are analogous notebooks for the other two model architectures, which facilitated training all three in parallel using multiple systems. **There are two important things to note about these notebooks:**

+ During the project there was an issue with the code for my initial cross-validation which led to some metrics not being logged properly. As such, even though I had completed cross-validation, I did not have a few of the metrics I wanted to put into the plots in the manuscript/final report (i.e. supplemental Figure 13). I re-ran cross-validation towards the end of the project on the same CV folds to get some of these metrics for the report. This is why the dates/timestamps for cross-validation in these files are later than the those of the final training run.

+ A few cell outputs in these notebooks stop mid-epoch. This was due to a bug where the Jupyter notebook would occasionally lose communication with the main process and stop updating, despite the fact that training continued successfully to the end (as indicated by fully completed logs of all epochs and corresponding model weights).

### Make and save the folds used for cross-validation comparison of models

In [ ]:
# import keras_models as my_models
# import keras_utility_classes as kutils
# from preparation_1 import structural_profiles
# from get_normalization_params_5 import load_json

# # Load the estimated class weights
# class_weights = f'{structural_profiles}class_weights.json'
# class_weights = load_json(class_weights, verbose=True)

# # # Initialize a trainer object
# trainer = kutils.TrainingRun(class_weights = class_weights,
#                              class_encoding = {'control_exons' : 0, 
#                                                'control_introns_intergenic' : 0, 
#                                                'control_introns_intron' : 0, 
#                                                'intron-exon' : 1, 
#                                                'exon-intron' : 2})

# trainer.get_CV_folds(N_folds=5, save_dir='./CV_folds/')

# BiLSTM cross validation

In [5]:
import keras_models as my_models

model_init = {'memory_units' : 128, 
              'bidirectional' : True, 
              'dense_layer_dims': [256]} 

model = my_models.LSTM_classifier((77,28), 3, **model_init, print_summary=True)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_4 (Bidirectional) │ (None, 256)            │       160,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 227,331 (888.01 KB)

 Trainable params: 227,331 (888.01 KB)

 Non-trainable params: 0 (0.00 B)

In [1]:
import keras_models as my_models
import keras_utility_classes as kutils
from preparation_1 import structural_profiles
from get_normalization_params_5 import load_json

# Load the estimated class weights
class_weights = f'{structural_profiles}class_weights.json'
class_weights = load_json(class_weights, verbose=True)

# Define the folder where training data will be stored
project_folder = './BiLSTM_best_cv/'
base_name = 'BiLSTM' # no underscores
kutils.check_make_dir(project_folder)

# # Initialize a trainer object
trainer = kutils.TrainingRun(class_weights = class_weights,
                             class_encoding = {'control_exons' : 0, 
                                               'control_introns_intergenic' : 0, 
                                               'control_introns_intron' : 0, 
                                               'intron-exon' : 1, 
                                               'exon-intron' : 2})

# Name constructor function for the model type to use (one found in keras_models) and define required hyperparameters
model_creator = my_models.LSTM_classifier
model_init = {'memory_units' : 128, 
              'bidirectional' : True,
              'dense_layer_dims': [256],
              'print_summary' : True} 

# Run the model training in 5-fold cross validation mode, saving results and best models from each run
trainer.execute_training_run(model_creator=model_creator, 
                             model_args=model_init,
                             saves_base_dir = project_folder, 
                             base_name = base_name,
                             batch_size=64, # 64 was ~30 per epoch
                             max_epochs=10,
                             patience=0, 
                             run_type='cross_validation',
                             N_folds=5, 
                             folds_dir='./CV_folds/',
                             lr_decay=False,
                             lr=0.001, # default learning rate # 1291 minutes - (3 hrs for 2 CV folds (# 711 min for 1 fold)
                             plot_results=False) # 4352 s/epoch 6 epochs to stop: ~6 hours total; 6084s/epoch w/ all metrics on ~5 to converge; ~10 hours/cv; ~13-15/full set

2025-09-30 20:15:02.206086: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759277702.436484     577 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759277702.506962     577 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759277703.058884     577 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759277703.058914     577 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759277703.058916     577 computation_placer.cc:177] computation placer alr

load_json: Loaded ./3_Physicochemical_Profiles/class_weights.json.
TrainingRun(): Loading neccesary .json and .npy data files...
write_json_pretty: Wrote ./BiLSTM_best_cv/training_call.json.
execute_training_run(): loading cross-validation splits from files in ./CV_folds/...
execute_training_run(): it appears that 3/5 have already been completed. Resuming on fold 4.


I0000 00:00:1759277771.440939     577 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4697 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1060, pci bus id: 0000:01:00.0, compute capability: 6.1


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 256)            │       160,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       160,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 227,331 (888.01 KB)

 Trainable params: 227,331 (888.01 KB)

 Non-trainable params: 0 (0.00 B)

execute_training_run(): fold 5/5 training will now commence!
Epoch 1/10
55984/55984 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - AUC_PR_Control: 0.9879 - AUC_PR_Exon-Intron: 0.4615 - AUC_PR_Intron-Exon: 0.5742 - AUC_ROC_Control: 0.8768 - AUC_ROC_Exon-Intron: 0.8542 - AUC_ROC_Intron-Exon: 0.9130 - F1_Score_0.1: 0.6551 - F1_Score_0.2: 0.6114 - F1_Score_0.3: 0.5710 - F1_Score_0.4: 0.5333 - F1_Score_0.5: 0.4968 - F1_Score_0.6: 0.4608 - F1_Score_0.7: 0.4238 - F1_Score_0.8: 0.3881 - F1_Score_0.9: 0.3504 - F1_Score_argmax: 0.4978 - FPR_Control: 0.7237 - FPR_Exon-Intron: 0.0227 - FPR_Intron-Exon: 0.0162 - Precision_Control: 0.9600 - Precision_Exon-Intron: 0.6995 - Precision_Intron-Exon: 0.6992 - Recall_Control: 0.9592 - Recall_Exon-Intron: 0.1941 - Recall_Intron-Exon: 0.2398 - Recall_at_90P_Control: 1.0000 - Recall_at_90P_Exon-Intron: 0.1667 - Recall_at_90P_Intron-Exon: 0.2178 - categorical_accuracy: 0.9538 - kl_divergence: 0.1743 - loss: 0.3625
Epoch 1: val_loss improved from None to 0.19966, saving 

# BiLSTM Final Training!

In [ ]:
import keras_models as my_models
import keras_utility_classes as kutils
from preparation_1 import structural_profiles
from get_normalization_params_5 import load_json

# Load the estimated class weights
class_weights = f'{structural_profiles}class_weights.json'
class_weights = load_json(class_weights, verbose=True)

# Define the folder where training data will be stored
project_folder = './BiLSTM_final/'
base_name = 'BiLSTM' # no underscores
kutils.check_make_dir(project_folder)

# # Initialize a trainer object
trainer = kutils.TrainingRun(class_weights = class_weights,
                             class_encoding = {'control_exons' : 0, 
                                               'control_introns_intergenic' : 0, 
                                               'control_introns_intron' : 0, 
                                               'intron-exon' : 1, 
                                               'exon-intron' : 2})

# Name constructor function for the model type to use (one found in keras_models) and define required hyperparameters
model_creator = my_models.LSTM_classifier
model_init = {'memory_units' : 128, 
              'bidirectional' : True,
              'dense_layer_dims': [256],
              'print_summary' : True} 

# Run the model training in 5-fold cross validation mode, saving results and best models from each run
trainer.execute_training_run(model_creator=model_creator, 
                             model_args=model_init,
                             saves_base_dir = project_folder, 
                             base_name = base_name,
                             batch_size=64, # 64 was ~30 per epoch
                             max_epochs=10,
                             patience=0, 
                             run_type='final_eval',
                             lr_decay=False,
                             lr=0.001, # default learning rate # 750 minutes to train final # 639 min forreal
                             plot_results=False) # 4352 s/epoch 6 epochs to stop: ~6 hours total; 6084s/epoch w/ all metrics on ~5 to converge; ~10 hours/cv; ~13-15/full set

2025-09-27 21:55:11.220448: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759024511.459020     552 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759024511.533392     552 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759024512.078704     552 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759024512.078773     552 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1759024512.078776     552 computation_placer.cc:177] computation placer alr

load_json: Loaded ./3_Physicochemical_Profiles/class_weights.json.
check_make_dir(): made ./BiLSTM_final/.
TrainingRun(): Loading neccesary .json and .npy data files...
write_json_pretty: Wrote ./BiLSTM_final/training_call.json.


I0000 00:00:1759024574.402891     552 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4697 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1060, pci bus id: 0000:01:00.0, compute capability: 6.1


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 256)            │       160,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 Trainable params: 227,331 (888.01 KB)

 Non-trainable params: 0 (0.00 B)

execute_training_run(): final training and evaluation will soon commence!
execute_training_run(): training will now commence!
Epoch 1/10


I0000 00:00:1759024592.956632     790 cuda_dnn.cc:529] Loaded cuDNN version 90300


69981/69981 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - AUC_PR_Control: 0.9893 - AUC_PR_Exon-Intron: 0.5236 - AUC_PR_Intron-Exon: 0.6089 - AUC_ROC_Control: 0.8905 - AUC_ROC_Exon-Intron: 0.8715 - AUC_ROC_Intron-Exon: 0.9194 - F1_Score_0.1: 0.6843 - F1_Score_0.2: 0.6485 - F1_Score_0.3: 0.6090 - F1_Score_0.4: 0.5727 - F1_Score_0.5: 0.5351 - F1_Score_0.6: 0.4966 - F1_Score_0.7: 0.4566 - F1_Score_0.8: 0.4132 - F1_Score_0.9: 0.3653 - F1_Score_argmax: 0.5361 - FPR_Control: 0.6936 - FPR_Exon-Intron: 0.0204 - FPR_Intron-Exon: 0.0153 - Precision_Control: 0.9617 - Precision_Exon-Intron: 0.7369 - Precision_Intron-Exon: 0.7002 - Recall_Control: 0.9624 - Recall_Exon-Intron: 0.2292 - Recall_Intron-Exon: 0.2617 - Recall_at_90P_Control: 1.0000 - Recall_at_90P_Exon-Intron: 0.2348 - Recall_at_90P_Intron-Exon: 0.2618 - categorical_accuracy: 0.9561 - kl_divergence: 0.1628 - loss: 0.3394
Epoch 1: val_loss improved from None to 0.15148, saving model to ./BiLSTM_final/BiLSTM_best.keras
69981/69981 ━━━━━━━━━━━━━━━━━━━